In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:

%pip install --quiet geopandas fiona

%pip install rasterio

%pip install pystac



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
import os
import csv

import json
import xml.etree.ElementTree as ET
import datetime
import requests
from io import BytesIO

from rasterio.warp import transform_bounds

from matplotlib.colors import ListedColormap, Normalize
from shapely.geometry import shape, mapping, MultiPolygon, Polygon,box
from IPython.display import Image, display


import numpy as np
import pystac
from pystac.extensions.table import TableExtension
from pystac import CatalogType
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension


### **Used ijson python package for reading the large GEOJSON files and get the metadata **

In [ ]:
pip install ijson

In [ ]:
import datetime
from pystac import Item
import pandas as pd
import geopandas as gpd
import os
import ijson
import json
from shapely.geometry import shape, mapping

In [ ]:
# #geojson_filepath = '/content/drive/MyDrive/Pan_india/pan_india_drainage_lines.geojson'
# geojson_filepath = '/content/drive/MyDrive/Pan_india/Microwatershed_boundries_v2.geojson'

# geojson_filepath = "/content/drive/MyDrive/PAN_india_vector_layerstest.csv"

#geojson_filepath = "/content/drive/MyDrive/PAN_india_layers_updated.csv"

geojson_filepath = "/content/drive/MyDrive/PAN_india_vector_layers_mapping.csv"

SUB_COLLECTIONS = {}

STAC_SAVE_DIR = "/content/drive/MyDrive/STAC_spec_PANIndia_test"
ROOT_CATALOG_HREF = os.path.join(STAC_SAVE_DIR, "catalog.json")
PANINDIA_COLLECTION_ID = "PanIndiaCatalogs"


In [ ]:
df = pd.read_csv(geojson_filepath)

print("DataFrame loaded successfully")
print(df.head())

In [ ]:
try:

    COLUMN_DESC_DF = pd.read_csv('/content/drive/MyDrive/column_description_9dec.csv')
    print("Column descriptions loaded successfully.")
except FileNotFoundError:
    print("WARNING: 'column_descriptions.csv' not found. Table extension will be empty.")
    COLUMN_DESC_DF = pd.DataFrame({'layer_name': [], 'column_name': [], 'column_name_description': []})

In [ ]:
def get_or_create_root_catalog():
    if os.path.exists(ROOT_CATALOG_HREF):
        try:
            print(f"Attempting to load existing root catalog from: {ROOT_CATALOG_HREF}")

            root_catalog = pystac.Catalog.from_file(ROOT_CATALOG_HREF)
            print("Found and loaded existing root catalog.")

            panindia_collection = root_catalog.get_child(PANINDIA_COLLECTION_ID)

            if panindia_collection:
                print(f"Found existing collection: {PANINDIA_COLLECTION_ID}.")
                return root_catalog, panindia_collection
            else:
                print(f"Root catalog found, but collection '{PANINDIA_COLLECTION_ID}' is missing. Proceeding to create/add it.")


        except Exception as e:
            print(f"Error loading existing catalog structure: {e}. Recreating from scratch...")


    print("Creating a new STAC structure.")

    root_catalog = pystac.Catalog(
        id="PANindia",
        title="STAC Catalog for PAN India Layers",
        description="Root catalog for PAN India assets and their metadata."
    )

    panindia_collection = pystac.Collection(
        id=PANINDIA_COLLECTION_ID,
        title="Pan India Spatio Temporal Asset Catalog",
        description="This spatio temporal asset catalog contains all data layers of CoRE Stack (https://core-stack.org/) generated at Pan India level.",
        extent=pystac.Extent(
            spatial=pystac.SpatialExtent([[68, 8, 98, 37]]),
            temporal=pystac.TemporalExtent([[datetime.datetime(2005, 1, 1), datetime.datetime(2024, 12, 31)]])
        ),
        license="CC-BY-4.0",
        providers=[
            pystac.Provider(
                name="CoREstack",
                roles=[pystac.ProviderRole.PRODUCER, pystac.ProviderRole.PROCESSOR, pystac.ProviderRole.HOST],
                url="https://core-stack.org/"
            )
        ],
        keywords=["social-ecological", "sustainability", "CoRE stack"]
    )

    root_catalog.add_child(panindia_collection)


    if not os.path.exists(STAC_SAVE_DIR):
        os.makedirs(STAC_SAVE_DIR)


    root_catalog.normalize_hrefs(STAC_SAVE_DIR)
    root_catalog.save(catalog_type=CatalogType.SELF_CONTAINED)
    print("Created and saved new Root Catalog and Pan India Collection.")

    return root_catalog, panindia_collection

In [ ]:
def create_sub_collection(title, parent_collection):

    title_str = str(title) if title is not None else "unknown"
    collection_id = title_str.lower().replace(' ', '-').replace(':', '').replace('/', '-')

    description = f"STAC collection for {title_str} of Pan India."

    if collection_id not in SUB_COLLECTIONS:
        print(f"Creating new Sub-Collection: {title_str}")
        sub_collection = pystac.Collection(
            id=collection_id,
            title=title_str,
            description=description,
            extent=parent_collection.extent,
            license=parent_collection.license,
            providers=parent_collection.providers
        )


        SUB_COLLECTIONS[collection_id] = sub_collection
    return SUB_COLLECTIONS[collection_id]


In [ ]:

def get_feature_geometry(filepath):
    ext = os.path.splitext(filepath)[1].lower()

    if ext in [".geojson", ".json"]:
        with open(filepath, 'r', encoding="utf-8") as f:
            for item in ijson.items(f, 'features.item', use_float=True):
                geom = item.get("geometry")
                shapely_geom = shape(geom)
                minx, miny, maxx, maxy = shapely_geom.bounds
                return geom, [minx, miny, maxx, maxy]


    if ext == ".shp":

        for enc in ["utf-8", "latin-1", "cp1252"]:
            try:
                gdf = gpd.read_file(filepath, encoding=enc)
                break
            except UnicodeDecodeError:
                continue


        first_geom = gdf.iloc[0].geometry
        geom = mapping(first_geom)
        bbox = gdf.total_bounds.tolist()

        return geom, bbox

    raise ValueError(f"Unsupported vector file format: {filepath}")


def get_first_feature_properties(filepath):
    ext = os.path.splitext(filepath)[1].lower()


    if ext in [".geojson", ".json"]:
        with open(filepath, 'r', encoding="utf-8") as f:
            for item in ijson.items(f, 'features.item.properties', use_float=True):
                properties_dict = {
                    key: {"value": value, "type": type(value).__name__}
                    for key, value in item.items()
                }
                break

        return {"properties": properties_dict}


    if ext == ".shp":

        for enc in ["utf-8", "latin-1", "cp1252"]:
            try:
                gdf = gpd.read_file(filepath, encoding=enc)
                break
            except UnicodeDecodeError:
                continue

        props_raw = gdf.iloc[0].to_dict()


        properties_dict = {
            key: {"value": value, "type": type(value).__name__}
            for key, value in props_raw.items()
        }

        return {"properties": properties_dict}

    raise ValueError(f"Unsupported vector file format: {filepath}")


In [ ]:

def load_layer_descriptions(csv_path="/content/drive/MyDrive/layer_descriptions.csv"):
    desc_map = {}
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:

            norm_key = (
                row["layer_name"]
                .lower()
                .replace(" ", "-")
                .replace(":", "")
                .replace("/", "-")
                .strip()
            )
            desc_map[norm_key] = row["layer_description"].strip()
    return desc_map

LAYER_DESCRIPTIONS = load_layer_descriptions("/content/drive/MyDrive/layer_descriptions.csv")


In [ ]:
def generate_vector_stac(layer_name, file_path, column_desc_df, drive_link=None):
    print(f"Processing vector layer: {layer_name}")

    item_id = (
        layer_name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .strip()
    )


    description_id = (
        layer_name.lower()
        .replace(" ", "-")
        .replace(":", "")
        .replace("/", "-")
        .strip()
    )


    item_description = LAYER_DESCRIPTIONS.get(description_id, layer_name)


    footprint, bbox = get_feature_geometry(file_path)
    properties_dict = get_first_feature_properties(file_path)["properties"]


    vector_gdf_dtypes = pd.DataFrame(
        [(key, value['type']) for key, value in properties_dict.items()],
        columns=['column_name', 'column_dtype']
    )

    try:

        vector_item = pystac.Item(
            id=item_id,
            geometry=footprint,
            bbox=bbox,
            datetime=datetime.datetime.now(datetime.timezone.utc),
            properties={"description": item_description}
        )

        # Projection extension
        proj_ext = ProjectionExtension.ext(vector_item, add_if_missing=True)
        proj_ext.epsg = 4326


        csv_layer_normalized = (
            column_desc_df["layer_name"]
            .str.lower()
            .str.replace(" ", "_")
            .str.replace("/", "_")
            .str.replace("-", "_")
            .str.strip()
        )


        vector_desc_filtered_df = column_desc_df[csv_layer_normalized == item_id]


        vector_merged_df = vector_gdf_dtypes.merge(
            vector_desc_filtered_df[['column_name', 'column_name_description']],
            on='column_name',
            how='left'
        ).fillna('')

        vector_merged_df.rename(
            columns={'column_name_description': 'column_description'},
            inplace=True
        )

        print(
            f"Schema merged for {layer_name}. "
            f"Found {len(vector_merged_df[vector_merged_df['column_description'] != ''])} column descriptions."
        )

        # Table extension
        table_ext = TableExtension.ext(vector_item, add_if_missing=True)
        table_ext.columns = [
            {
                "name": row['column_name'],
                "type": str(row['column_dtype']),
                "description": row['column_description']
            }
            for idx, row in vector_merged_df.iterrows()
        ]


        if drive_link:
            vector_item.add_asset(
                "drive-link",
                pystac.Asset(
                    href=drive_link,
                    media_type=pystac.MediaType.TEXT,
                    roles=["source"],
                    title="Vector File"
                )
            )
        else:
            print(f"No Drive link found for {layer_name}, skipping asset attachment.")

        return vector_item

    except Exception as e:
        print(f"Error generating STAC Item for {layer_name}: {e}")
        return None


In [ ]:
def run_stac_generation(df, panindia_collection, root_catalog, column_desc_df):
    generated_items_count = 0

    created_sub_collections = {}

    print(f"Starting generation for {len(df)} layers...")

    for idx, row in df.iterrows():

        coll_name = row["collection_name"]
        display_name = row["display_name"]
        item_id = row["layer_name"]
        file_path = row["asset link"]
        drive_link = row["drive_link"]


        if coll_name not in created_sub_collections:
            sub_coll = create_sub_collection(coll_name, panindia_collection)

            if sub_coll.id not in [child.id for child in panindia_collection.get_children()]:
                panindia_collection.add_child(sub_coll)
            created_sub_collections[coll_name] = sub_coll
            print(f"\nGroup: {coll_name}")

        current_collection = created_sub_collections[coll_name]


        if current_collection.get_item(item_id):
            print(f"Skipping: {item_id} already exists.")
            continue


        if os.path.exists(file_path):

            vector_item = generate_vector_stac(display_name, file_path, column_desc_df, drive_link)

            if vector_item:

                vector_item.id = item_id
                current_collection.add_item(vector_item)
                generated_items_count += 1
                print(f"Added Item: {item_id} to {current_collection.id}")

                if generated_items_count % 3 == 0:
                    root_catalog.normalize_hrefs(STAC_SAVE_DIR)
                    root_catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
        else:
            print(f"Warning: File not found for {display_name} at {file_path}")

    save_catalog(root_catalog, generated_items_count)

In [ ]:
def save_catalog(root_catalog, count):

    if count >= 0:

        root_catalog.normalize_hrefs(STAC_SAVE_DIR)

        root_catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
        print(f"STAC catalog saved.")
        print(f"Total items generated:{count}")
    else:
        print("No items were generated.")

In [ ]:

root_catalog, panindia_collection = get_or_create_root_catalog()

run_stac_generation(df, panindia_collection, root_catalog, COLUMN_DESC_DF)